<a href="https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jjcodes999/flyrank-ml-repo/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Answer:

The paper (*The State of AI-Driven SEO in Numbers*, March 2026; 341,701 pages, 57 brands) leads with direct aggregate comparisons, places ML in an exploratory appendix, and states that it is observational. My questions below are suggestions for the next level of rigor, the same questions I then apply to my own model.

### Finding A: "Click Capture by Position Tier" (Finding #3, p. 8)
**The claim:** weighted CTR (total clicks ÷ total impressions per tier) falls as pages move down the results: Top 3 0.423%, Page 1 (4–10) 0.339%, Striking (11–20) 0.325%, Page 3–5 0.163%, Deep 0.050%. The paper already improved on an older per-row average, and says these are not generic Google benchmarks.

**Where the numbers come from:** there is no model label here. It is a descriptive ratio pooled across all 57 brands, with each page placed in a tier by its average position over the 90-day window.

**My methodology questions:**
1. **Does the pattern hold inside each brand?** Because clicks and impressions are pooled, the largest brands carry the most weight in every tier. Showing the median of per-brand CTRs next to the pooled number, plus the largest brand's share of impressions per tier, would show whether the curve is a portfolio-wide pattern or driven by a few big brands. (In my own March data, one client filled about half of my top-100 list, so this risk is real; my check is below.)

**What my own data shows** (check below; March 2026, 37 clients, pages with at least 100 impressions, 1–15 March): the pooled weighted CTR follows almost the same curve as the paper (Top 3 0.437%, Page 1 0.330%, Striking 0.336%, Page 3–5 0.141%, Deep 0.038%). But pooling hides differences between clients. Pooled, Page 1 and Striking look equal (0.330% vs 0.336%), while the typical (median) client shows a clear drop (0.368% → 0.245%). In the Page 3–5 tier one client supplies 73% of all impressions, and the largest client supplies 37–45% of impressions in every other tier. This is one month of a different dataset, not the paper's, but it supports asking for a per-brand view next to the pooled curve.

2. **Is Page 1 vs Striking a real difference?** 0.339% and 0.325% are very close, and the paper reports no confidence intervals (p. 36). Page counts per tier and an uncertainty range would tell the reader whether these two tiers actually differ.
3. **How do the definitions compare?** The repo's data dictionary quotes about 2.78% CTR for positions 1–3 at warehouse scale, while the paper's Top-3 figure is 0.423%. A one-line note on what differs (population, window, or how position is averaged) would help readers reconcile them.

### Finding B: "What Predicts Growth?" (ML appendix, p. 29)
**The claim:** a logistic regression separates growing from declining pages with 71% holdout accuracy. The paper correctly calls this descriptive and exploratory.

**Where the label comes from:** trend direction, computed from impressions in the last 30 days versus the 30 days before (Up / Down = more than 10% change, p. 5). The model uses an 80/20 split (p. 36) on 61.8K active pages.

**My methodology questions:**
1. **Do any features overlap the label window?** If features such as impressions, clicks or days visible are 90-day totals, they include the same 30 days the label is computed from, so the model could partly read the outcome. Building features only from the days *before* the label window would remove that risk.
2. **What is the naive baseline?** Finding #1 counts 74,187 rising vs 45,272 falling pages (about 62% rising). If the ML sample has a similar mix, always guessing "growing" would already reach about 62%, so 71% would be roughly 9 points of skill. Printing the majority-class accuracy next to 71% would make the result easy to judge.
3. **Were brands kept separate between training and testing?** An 80/20 split of pages can place pages from the same brand on both sides, which can make accuracy look higher than it would be on a new brand. A brand-grouped split would show how well the model carries over to brands it has not seen.

In [1]:
%pip -q install duckdb

import duckdb
import numpy as np
import pandas as pd
import sklearn
from google.colab import userdata

# Connect to the warehouse
HF_TOKEN = userdata.get("HF_TOKEN")
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Same frame as ML-04 / ML-07 / ML-08: one row per page. Clues 1-15 March, answer 16-31 March.
pages = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') AS impressions_h1,
        SUM(gsc_clicks)      FILTER (WHERE report_date <= DATE '2026-03-15') AS clicks_h1,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h1,
        COUNT(DISTINCT report_date) FILTER (WHERE report_date <= DATE '2026-03-15' AND gsc_impressions > 0) AS days_visible_h1,
        SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15') AS impressions_h2,
        SUM(gsc_clicks)      FILTER (WHERE report_date > DATE '2026-03-15') AS clicks_h2,
        SUM(gsc_avg_position * gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0)
          / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-15' AND gsc_avg_position > 0), 0) AS position_h2
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING impressions_h1 >= 100 AND impressions_h2 >= 50
""").df()
pages = pages.dropna(subset=["position_h1", "position_h2"]).copy()
pages = pages.sort_values(["client_hash_id", "content_hash_id"]).reset_index(drop=True)  # fixed row order -> reproducible

bands = [0, 3, 5, 7, 10, 15, 20, 30, 50, 1000]
pages["ctr_h1"] = pages["clicks_h1"] / pages["impressions_h1"] * 100
pages["band_h1"] = pd.cut(pages["position_h1"], bands)
pages["expected_ctr_h1"] = pages.groupby("band_h1", observed=True)["ctr_h1"].transform("median")
pages["missed_clicks_h1"] = pages["impressions_h1"] * (pages["expected_ctr_h1"] - pages["ctr_h1"]) / 100

pages["ctr_h2"] = pages["clicks_h2"] / pages["impressions_h2"] * 100
pages["band_h2"] = pd.cut(pages["position_h2"], bands)
pages["expected_ctr_h2"] = pages.groupby("band_h2", observed=True)["ctr_h2"].transform("median")
pages["still_under"] = (pages["ctr_h2"] < pages["expected_ctr_h2"]).astype(int)

# Frozen Week-4 baseline rule
MIN_IMPRESSIONS, MAX_POSITION = 1000, 20
flagged = (pages["impressions_h1"] >= MIN_IMPRESSIONS) & (pages["position_h1"] <= MAX_POSITION) & (pages["ctr_h1"] < pages["expected_ctr_h1"])
pages["baseline_score"] = np.where(flagged, pages["missed_clicks_h1"], 0.0)
FEATURES = ["impressions_h1", "ctr_h1", "position_h1", "days_visible_h1", "missed_clicks_h1"]

print("Pages:", len(pages), "| clients:", pages["client_hash_id"].nunique(),
      "| base rate:", round(pages["still_under"].mean(), 2), "| scikit-learn", sklearn.__version__)

# CHECK OF QUESTION A1 ON MY OWN DATA: pooled CTR vs the typical client's CTR, per position tier (1-15 March)
tier = pd.cut(pages["position_h1"], [0, 3, 10, 20, 50, 1000],
              labels=["Top 3 (1-3)", "Page 1 (4-10)", "Striking (11-20)", "Page 3-5 (21-50)", "Deep (50+)"])
by_client = (pages.assign(tier=tier)
             .groupby(["tier", "client_hash_id"], observed=True)
             .agg(clicks=("clicks_h1", "sum"), impressions=("impressions_h1", "sum")))
by_client["ctr"] = by_client["clicks"] / by_client["impressions"] * 100
g = by_client.groupby("tier", observed=True)
check = pd.DataFrame({
    "pooled_weighted_ctr_pct": g["clicks"].sum() / g["impressions"].sum() * 100,
    "median_client_ctr_pct": g["ctr"].median(),
    "clients": g.size(),
    "biggest_client_share_of_impressions": g["impressions"].apply(lambda s: s.max() / s.sum()),
}).round(3)
display(check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Pages: 75817 | clients: 37 | base rate: 0.35 | scikit-learn 1.6.1


,pooled_weighted_ctr_pct,median_client_ctr_pct,clients,biggest_client_share_of_impressions
tier,,,,
Top 3 (1-3),0.437,0.500,23,0.407
Page 1 (4-10),0.330,0.368,37,0.372
Striking (11-20),0.336,0.245,32,0.367
Page 3-5 (21-50),0.141,0.130,26,0.726
Deep (50+),0.038,0.033,21,0.446


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

Answer:

I re-ran my Week-5 models (same 5 features, same label, same seeds) under two splits and scored every method on the same folds. **BEFORE:** a random 5-fold split of pages; 30–34 of the 37 clients appear in both training and testing in every fold. **AFTER:** my grouped 5-fold split by client; no client appears on both sides.

| Split | Method | P@10 | P@20 | P@50 | P@100 | AUC |
|---|---|---|---|---|---|---|
| Before (random) | Base rate | 0.355 | 0.355 | 0.355 | 0.355 | 0.500 |
| | Week-4 baseline rule | 0.860 | 0.910 | 0.944 | 0.912 | 0.583 |
| | Logistic regression | 0.900 | 0.900 | **0.948** | **0.926** | 0.822 |
| | Random forest | 0.980 | 0.980 | 0.932 | 0.912 | 0.824 |
| **After (grouped)** | Base rate | 0.351 | 0.351 | 0.351 | 0.351 | 0.500 |
| | Week-4 baseline rule | 0.940 | 0.930 | **0.924** | **0.898** | 0.566 |
| | Logistic regression | 0.940 | 0.920 | 0.896 | 0.876 | 0.809 |
| | Random forest | 0.980 | 0.980 | 0.920 | 0.884 | 0.805 |

AUC = how often a page that stayed under-performing is ranked above one that did not (0.5 = coin flip, 1.0 = always). The baseline's AUC is low by design: it gives a score of 0 to every page it does not flag, so it only orders the top of the list.

**What the before/after shows:**
- Both models score a little lower under the honest split: AUC falls from 0.822 to 0.809 (logistic regression) and from 0.824 to 0.805 (random forest), and Precision@100 falls by 3–5 points. The random split let the models gain a little from having seen the same clients in training.
- The gap is small at the very top (the forest's P@10 and P@20 stay at 0.98) and larger deeper in the list.
- **It changes one conclusion.** Under the random split, logistic regression appears to beat the baseline deeper in the list (P@50 0.948 vs 0.944; P@100 0.926 vs 0.912). Under the honest split it does not (0.896 vs 0.924; 0.876 vs 0.898). A random split would have led me to over-credit the learned model.
- The baseline rule does not learn, so its own changes between the two splits (for example P@10 0.86 → 0.94) come only from which pages land in each test fold. That shows how much these numbers move by chance alone, so differences of a few points should not be over-read.

**Failure examples (random forest, grouped split; tables below):**
- **False alarms: 2 of 100 top-20 picks** were not still under-performing later. Both had near-zero CTR at a top-3 position (1 click from 8,975 impressions at position 2.7; 4 clicks from 5,710 impressions at position 2.4) and were also flagged by my baseline. They looked like the clearest problems, yet recovered in 16–31 March. My data cannot tell whether the page moved position, gained a few clicks, or had a first-half tracking glitch.
- **Misses: 5,869 pages** stayed under-performing but were ranked in the bottom half. The five with the most missed clicks are mild cases: CTR only modestly below typical (for example 0.18% vs 0.25% at position 4.1), worth about 2 missed clicks each, so for a review queue they matter little. All five were flagged by my baseline rule, so the rule and the forest disagree on them. One has an average position of 0.9, which is impossible (Google positions start at 1), the same data-quality issue I found in ML-07.

In [2]:
# BEFORE vs AFTER: the same models, scored under a random split and under the honest grouped split
from sklearn.model_selection import KFold, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

def squash(df):
    """For logistic regression only: log-squash the two very skewed features."""
    X = df[FEATURES].copy()
    X["impressions_h1"] = np.log1p(X["impressions_h1"])
    X["missed_clicks_h1"] = np.sign(X["missed_clicks_h1"]) * np.log1p(np.abs(X["missed_clicks_h1"]))
    return X

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores), kind="stable")
    return np.asarray(labels)[order[:k]].mean()

KS = [10, 20, 50, 100]
SPLITS = {
    "BEFORE: random 5-fold": list(KFold(n_splits=5, shuffle=True, random_state=42).split(pages)),
    "AFTER: grouped 5-fold by client": list(GroupKFold(n_splits=5).split(pages, groups=pages["client_hash_id"])),
}

for name, folds in SPLITS.items():
    shared = [len(set(pages.iloc[tr]["client_hash_id"]) & set(pages.iloc[te]["client_hash_id"])) for tr, te in folds]
    print(f"{name}: clients in BOTH train and test, per fold = {shared}")

rows = []
pages["rf_score"] = np.nan
for split_name, folds in SPLITS.items():
    for i, (train_idx, test_idx) in enumerate(folds, 1):
        train, test = pages.iloc[train_idx], pages.iloc[test_idx]
        lr = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(squash(train), train["still_under"])
        rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1).fit(train[FEATURES], train["still_under"])
        scores = {"Week-4 baseline rule": test["baseline_score"].values,
                  "Logistic regression": lr.predict_proba(squash(test))[:, 1],
                  "Random forest": rf.predict_proba(test[FEATURES])[:, 1]}
        if split_name.startswith("AFTER"):
            pages.loc[test.index, "rf_score"] = scores["Random forest"]   # kept for the failure examples below
        y = test["still_under"].values
        rows.append({"split": split_name, "method": "Random pick (base rate)", **{f"P@{k}": y.mean() for k in KS}, "AUC": 0.5})
        for m, s in scores.items():
            rows.append({"split": split_name, "method": m, **{f"P@{k}": precision_at_k(s, y, k) for k in KS}, "AUC": roc_auc_score(y, s)})
    print(split_name, "- done")

res = pd.DataFrame(rows)
print("\nAverage over 5 folds:")
display(res.groupby(["split", "method"], sort=False).mean(numeric_only=True).round(3))

BEFORE: random 5-fold: clients in BOTH train and test, per fold = [30, 31, 31, 32, 34]
AFTER: grouped 5-fold by client: clients in BOTH train and test, per fold = [0, 0, 0, 0, 0]
BEFORE: random 5-fold - done
AFTER: grouped 5-fold by client - done

Average over 5 folds:


P@10   P@20   P@50  \
split                           method                                         
BEFORE: random 5-fold           Random pick (base rate)  0.355  0.355  0.355   
                                Week-4 baseline rule     0.860  0.910  0.944   
                                Logistic regression      0.900  0.900  0.948   
                                Random forest            0.980  0.980  0.932   
AFTER: grouped 5-fold by client Random pick (base rate)  0.351  0.351  0.351   
                                Week-4 baseline rule     0.940  0.930  0.924   
                                Logistic regression      0.940  0.920  0.896   
                                Random forest            0.980  0.980  0.920   

                                                         P@100    AUC  
split                           method                                 
BEFORE: random 5-fold           Random pick (base rate)  0.355  0.500  
                                Week-4 baseline rule     0.912  0.583  
                                Logistic regression      0.926  0.822  
                                Random forest            0.912  0.824  
AFTER: grouped 5-fold by client Random pick (base rate)  0.351  0.500  
                                Week-4 baseline rule     0.898  0.566  
                                Logistic regression      0.876  0.809  
                                Random forest            0.884  0.805

In [3]:
# FAILURE EXAMPLES under the honest (grouped) split - random forest
grouped_folds = SPLITS["AFTER: grouped 5-fold by client"]
false_alarms, misses = [], []
for i, (train_idx, test_idx) in enumerate(grouped_folds, 1):
    ranked = pages.iloc[test_idx].sort_values("rf_score", ascending=False)
    top20 = ranked.head(20)
    false_alarms.append(top20[top20["still_under"] == 0].assign(fold=i))      # flagged, but fine later
    bottom_half = ranked.iloc[len(ranked) // 2:]
    misses.append(bottom_half[bottom_half["still_under"] == 1].assign(fold=i)) # ranked low, but stayed bad

cols = ["fold", "impressions_h1", "clicks_h1", "ctr_h1", "expected_ctr_h1", "position_h1",
        "days_visible_h1", "missed_clicks_h1", "rf_score", "baseline_score"]
fa, mi = pd.concat(false_alarms), pd.concat(misses)

print(f"A) FALSE ALARMS - in the forest's top 20 per fold, but NOT still under-performing later: {len(fa)} of 100")
display(fa[cols].round(3))
print(f"\nB) MISSES - stayed under-performing, but the forest ranked them in the bottom half: {len(mi)} pages")
print("   The 5 with the most missed clicks:")
display(mi.nlargest(5, "missed_clicks_h1")[cols].round(3))

A) FALSE ALARMS - in the forest's top 20 per fold, but NOT still under-performing later: 2 of 100


,fold,impressions_h1,clicks_h1,ctr_h1,expected_ctr_h1,position_h1,days_visible_h1,missed_clicks_h1,rf_score,baseline_score
47856,1,8975.0,1.0,0.011,0.246,2.680,15,21.036,0.987,21.036
66014,5,5710.0,4.0,0.070,0.246,2.393,15,10.019,0.938,10.019



B) MISSES - stayed under-performing, but the forest ranked them in the bottom half: 5869 pages
   The 5 with the most missed clicks:


,fold,impressions_h1,clicks_h1,ctr_h1,expected_ctr_h1,position_h1,days_visible_h1,missed_clicks_h1,rf_score,baseline_score
39912,1,4720.0,2.0,0.042,0.089,14.807,15,2.192,0.363,2.192
40860,1,3473.0,1.0,0.029,0.089,13.248,15,2.084,0.402,2.084
24581,2,2847.0,5.0,0.176,0.245,4.143,15,1.962,0.420,1.962
28918,2,2824.0,5.0,0.177,0.245,4.154,15,1.906,0.427,1.906
34469,2,1994.0,3.0,0.150,0.246,0.915,15,1.896,0.487,1.896


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Answer:

I ran the attack checklist from the `hunting-leakage-and-validating` skill on my final five features (code and outputs below).

| Check | Result |
|---|---|
| Timeline: all features before the label window | ✅ All five features come from 1–15 March; the label comes from 16–31 March. |
| Label-derived or sibling columns in the features | ✅ None. |
| Product flags / existing scores as features | ✅ None (the dataset ships none). |
| Harness check: add a deliberate leak | ✅ Adding `ctr_gap_h2` (built from the label window) raises AUC from 0.805 to 1.000 and P@20 / P@100 to 1.00, so the test setup does detect leakage. The column was then removed. |
| Train without suspect features | ✅ No collapse. Without `position_h1`: AUC 0.799, P@100 0.866. Without `missed_clicks_h1`: AUC 0.806, P@20 0.94. The model does not depend on one suspicious column. |
| Split grouped by client | ✅ 0 clients shared between training and testing (Section 2). |
| Base rate printed next to every metric | ✅ |
| Metrics computed out-of-fold | ✅ Every score comes from clients the model did not train on. |
| Population selection uses the outcome window | ⚠️ Yes, slightly (see below). |

**What I found: my population uses the outcome window.** My frame keeps only pages with at least 50 impressions in 16–31 March, the same window as my label. Of 77,540 pages with at least 100 impressions in 1–15 March, 1,723 (2.2%) were dropped by this rule. At the decision moment (16 March) I could not have known to exclude them. The share is small and the filter is needed to compute a stable second-half CTR, but it is a choice I disclose: my results describe pages that stayed visible in the second half, not pages whose impressions fell below 50.

**Smaller notes:**
- **The baseline had a small head start.** My rule's 1,000-impression cutoff was chosen in ML-07 by looking at the label on all pages, including the clients that later serve as test folds. Its scores in the comparison are therefore slightly optimistic. A stricter test would choose the cutoff inside each training fold.
- The "typical CTR" for each position band is computed from all pages' first-half data, including test-fold pages. It uses no label information, but a stricter version would compute it from training clients only.
- Position was the top feature in ML-08, yet removing it barely changes the scores (AUC 0.805 → 0.799), because missed clicks already carries position information through the band's typical CTR. Position is not secretly carrying the answer.

In [4]:
# LEAKAGE AUDIT on my final feature set (random forest, honest grouped split)

# 1) Timeline: every feature must come from BEFORE the label window
timeline = pd.DataFrame({
    "column": FEATURES + ["still_under (label)"],
    "comes_from": ["1-15 March (before the decision)"] * len(FEATURES) + ["16-31 March (after the decision)"],
})
print("1) TIMELINE")
display(timeline)
print("Features taken from the label window:", [f for f in FEATURES if f.endswith("_h2")] or "none")
print("Product flags / existing scores among features:",
      [f for f in FEATURES if any(w in f for w in ["health", "priority", "flag", "action"])] or "none")

# 2) Train WITH a deliberate leak, and WITHOUT each suspect feature
def grouped_rf(features):
    aucs, p20s, p100s = [], [], []
    for train_idx, test_idx in grouped_folds:
        train, test = pages.iloc[train_idx], pages.iloc[test_idx]
        rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=42, n_jobs=-1)
        rf.fit(train[features], train["still_under"])
        s, y = rf.predict_proba(test[features])[:, 1], test["still_under"].values
        aucs.append(roc_auc_score(y, s)); p20s.append(precision_at_k(s, y, 20)); p100s.append(precision_at_k(s, y, 100))
    return {"AUC": np.mean(aucs), "P@20": np.mean(p20s), "P@100": np.mean(p100s)}

final = res[res["split"].str.startswith("AFTER") & (res["method"] == "Random forest")][["AUC", "P@20", "P@100"]].mean().to_dict()
pages["ctr_gap_h2"] = pages["expected_ctr_h2"] - pages["ctr_h2"]          # LEAK ON PURPOSE (built from the label window)
variants = {
    "Final model (5 features)": final,
    "+ leaked ctr_gap_h2 (harness check)": grouped_rf(FEATURES + ["ctr_gap_h2"]),
    "- without position_h1": grouped_rf([f for f in FEATURES if f != "position_h1"]),
    "- without missed_clicks_h1": grouped_rf([f for f in FEATURES if f != "missed_clicks_h1"]),
}
pages = pages.drop(columns=["ctr_gap_h2"])                                # leak removed again
print("\n2) TRAIN-WITH / TRAIN-WITHOUT TESTS (random forest, grouped split)")
display(pd.DataFrame(variants).T.round(3))
print("Leaked column removed again:", "ctr_gap_h2" not in pages.columns)

# 3) Population check: does choosing WHICH pages to include use the outcome window?
pop = con.sql(f"""
    SELECT COUNT(*)                                           AS pages_with_100_impr_in_h1,
           COUNT(*) FILTER (WHERE imp_h2 >= 50)               AS kept_also_50_impr_in_h2,
           COUNT(*) FILTER (WHERE COALESCE(imp_h2, 0) < 50)   AS dropped_by_h2_filter
    FROM (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-15') AS imp_h1,
               SUM(gsc_impressions) FILTER (WHERE report_date >  DATE '2026-03-15') AS imp_h2
        FROM {MARCH}
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
    )
    WHERE imp_h1 >= 100
""").df()
pop["share_dropped_pct"] = (pop["dropped_by_h2_filter"] / pop["pages_with_100_impr_in_h1"] * 100).round(1)
print("\n3) POPULATION CHECK - does choosing WHICH pages to include use the outcome window?")
display(pop)

1) TIMELINE


,column,comes_from
0,impressions_h1,1-15 March (before the decision)
1,ctr_h1,1-15 March (before the decision)
2,position_h1,1-15 March (before the decision)
3,days_visible_h1,1-15 March (before the decision)
4,missed_clicks_h1,1-15 March (before the decision)
5,still_under (label),16-31 March (after the decision)


Features taken from the label window: none
Product flags / existing scores among features: none

2) TRAIN-WITH / TRAIN-WITHOUT TESTS (random forest, grouped split)


,AUC,P@20,P@100
Final model (5 features),0.805,0.98,0.884
+ leaked ctr_gap_h2 (harness check),1.000,1.00,1.000
- without position_h1,0.799,0.98,0.866
- without missed_clicks_h1,0.806,0.94,0.868


Leaked column removed again: True

3) POPULATION CHECK - does choosing WHICH pages to include use the outcome window?


,pages_with_100_impr_in_h1,kept_also_50_impr_in_h2,dropped_by_h2_filter,share_dropped_pct
0,77540,75817,1723,2.2


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Answer:

I went back through my own notebooks and rewrote the three sentences that said more than my evidence shows. The code below recomputes the numbers behind the rewrites.

**1. ML-08 (model notebook)**
> *Original:* "For an editor with time for 20 pages, the baseline's list points at about twice the potential clicks."

*Problem:* "potential clicks" suggests these clicks could be won back by editing. Missed clicks only measure a gap: impressions × (typical CTR for the position band − the page's CTR). Nothing in my data tests whether editing closes that gap.

*Rewrite:* "In March 2026 data, the baseline's top 20 per fold covered about twice the estimated click gap of the random forest's top 20 (about 755 vs 362 missed clicks, averaged over 5 client-grouped folds). Whether editing these pages would recover any of that gap was not tested."

**2. ML-03 (task framing)**
> *Original:* "So a high score shows the flags are real and lasting, not random noise."

*Problem:* "real" and "not random noise" are stronger than one check on one data slice can support. The check showed persistence for one month on high-traffic pages, and its position window partly overlapped the checking period.

*Rewrite:* "In the starter data, the 20 pages with the most missed clicks in the first 60 days were all still below typical CTR in the next 30 days (Precision@20 = 1.00 vs a base rate of 0.42). This is a measured sign that low CTR persisted for a month on these high-traffic pages, on one data slice whose position window partly overlaps the checking period."

**3. ML-07 (baseline): the action label**
> *Original:* every flagged page gets the action `review_title_and_meta`.

*Problem:* the label implies that the title or meta description is the cause. My audits found other likely causes: 19% of flagged pages had zero clicks (possible tracking issues), 194 had an impossible average position below 1, and one client fills about half of the top of the list (a possible site-wide cause).

*Rewrite:* "Flagged pages had a lower CTR than typical for their position in 1–15 March, and most stayed below typical in 16–31 March. This is decision-support: a title and meta review is one possible action, alongside a tracking check for zero-click pages and a site-level review when one client dominates. The data does not show what causes the low CTR."

**My safe one-line summary of the project so far:**
"On March 2026 warehouse data, a transparent position-and-volume rule ranked pages whose CTR stayed below typical for their position at Precision@20 of 0.93 across 5 client-grouped folds (base rate 0.35; slightly optimistic, because the rule's cutoff was chosen on the same month). Learned models were slightly more precise at the very top but not deeper in the list, and nothing here shows that editing these pages would raise their clicks."

In [5]:
# EVIDENCE BEHIND THE REWRITTEN CLAIMS (computed live, honest grouped split)
rows = []
for i, (train_idx, test_idx) in enumerate(grouped_folds, 1):
    test = pages.iloc[test_idx]
    for name, col in [("Week-4 baseline rule", "baseline_score"), ("Random forest", "rf_score")]:
        top = test.sort_values(col, ascending=False).head(20)
        rows.append({"method": name,
                     "missed_clicks_in_top20": top["missed_clicks_h1"].clip(lower=0).sum(),
                     "precision_at_20": top["still_under"].mean()})
print("Claims 1 and summary - top 20 per fold, averaged over 5 client-grouped folds:")
display(pd.DataFrame(rows).groupby("method").mean().round(2))

flagged_pages = pages[pages["baseline_score"] > 0]
print("\nClaim 3 - pages flagged by my rule:", len(flagged_pages))
print(f"  with 0 clicks in 1-15 March: {(flagged_pages['clicks_h1'] == 0).sum()} ({(flagged_pages['clicks_h1'] == 0).mean():.0%})")
print(f"  with an impossible average position below 1: {(flagged_pages['position_h1'] < 1).sum()}")

Claims 1 and summary - top 20 per fold, averaged over 5 client-grouped folds:


,missed_clicks_in_top20,precision_at_20
method,,
Random forest,361.81,0.98
Week-4 baseline rule,754.66,0.93



Claim 3 - pages flagged by my rule: 8710
  with 0 clicks in 1-15 March: 1676 (19%)
  with an impossible average position below 1: 194


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.